[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/01_%E6%8F%8F%E8%BF%B0%E7%B5%B1%E8%A8%88%E8%88%87%E6%A2%9D%E4%BB%B6%E6%A9%9F%E7%8E%87.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：「平均表現不錯」等於「每次都差不多」嗎？被模型標記的產品，真的有問題嗎？**

- **用在流程哪裡**：資料探索（描述一組數字）／評估解讀（讀懂分類結果）。
- **讀完要能做到**：區分中心與分散，會用計數表算條件機率，能解釋基準比例如何翻轉結論。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[數據結構](../名詞解釋/02_數據結構/README.md)、[評估指標](../名詞解釋/09_評估指標/README.md)

# 01｜從描述資料到更新機率

描述統計回答「這組數字長什麼樣子」，條件機率回答「知道一件事後，另一件事的機率怎麼變」。

**一、描述一組數字：中心與分散**

- **平均數**：$\bar{x} = \frac{1}{n}\sum x_i$。把總量平分給每一筆，所以**一筆極端值就會把它拉走**（本篇 90 分鐘延遲就是示範）。
- **中位數**：排序後正中間的值，不受極端值影響，看「典型一筆」用它。
- **變異數與標準差**：$\sigma^2$ 是與平均距離平方的平均，$\sigma$ 開根號後**單位回到原始單位**（分鐘）。本篇描述整組範例用 `ddof=0`；若要用樣本估計母體才用 `ddof=1`。
- **箱形圖怎麼看**：盒子是中間 50%（Q1～Q3），線是中位數，鬚延伸到 1.5 倍 IQR 內最遠的實際點——**鬚不一定是最大／最小值**，外面的點要先查證再決定去留。

**二、更新機率：條件機率與貝氏想法**

- **條件機率**：$P(A|B) = \frac{P(A \cap B)}{P(B)}$，白話是「**把分母換成已知條件**」。被標記才算分母，不能再用全體一萬件當分母。
- **貝氏更新**：$P(瑕疵|標記) = \frac{P(標記|瑕疵)\,P(瑕疵)}{P(標記)}$。$P(標記|瑕疵)=90\%$（recall 高）回答的是「有病抓到」；我們要的是反過來「抓到有病」$P(瑕疵|標記)$，兩者被**基準比例**（瑕疵只有 1%）徹底改變——這就是**基準率謬誤**，也是 precision 與 recall 常被搞混的原因。
- **生活比喻**：平均數像全班分豬肉，一個大胃王就讓每人份暴增；條件機率像篩檢，罕見病就算檢驗很準，陽性也不等於得病，要看發病率。

本篇全部使用教學合成數字，不代表任何真實模型成效。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib import font_manager
from IPython.display import display

# 本機優先使用教材字型；Colab 只下載這一個檔案。
candidates = [Path('ChineseFont.ttf'), Path('../ChineseFont.ttf'),
              Path('機率統計/ChineseFont.ttf')]
font_path = next((p for p in candidates if p.is_file()), None)
if font_path is None:
    font_path = Path('ChineseFont.ttf')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/ChineseFont.ttf',
        font_path)
font_manager.fontManager.addfont(str(font_path))
font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
sns.set_theme(style='whitegrid', font=font_name, rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：', font_name)

## 01｜平均數不等於典型的一筆

**為什麼做**：用配送時間比較一般情況與一筆延遲，理解平均數對極端值的敏感性。

**怎麼看**：同樣的前五筆加上 90 分鐘延遲後，平均數與中位數如何變化？箱形圖的鬚不是必然的最大／最小值。

In [ ]:
regular = np.array([18, 19, 20, 21, 22], dtype=float)
delayed = np.append(regular, 90)
summary = pd.DataFrame([
    {'情境': name, '平均數': values.mean(), '中位數': np.median(values),
     '標準差（ddof=0）': values.std(ddof=0)}
    for name, values in [('一般配送', regular), ('包含延遲', delayed)]
])
display(summary.round(2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([regular, delayed], tick_labels=['一般配送', '包含延遲'],
           patch_artist=True, boxprops={'facecolor': '#DFEAE3'},
           medianprops={'color': ORANGE, 'linewidth': 2})
ax.set(title='一筆延遲，會怎麼影響典型配送時間？', ylabel='配送時間（分鐘）')
plt.show()

## 02｜平均相同，風險可能不同

**為什麼做**：平均值描述中心，標準差描述離中心的分散程度；單位仍是分鐘。

**怎麼看**：兩組平均都為 20，但其中一組的快慢差異更大。這裡描述整組示例用 ddof=0；估計母體變異數常用 ddof=1。

In [ ]:
steady = np.array([18, 19, 20, 21, 22])
variable = np.array([5, 10, 20, 30, 35])
fig, ax = plt.subplots(figsize=(8, 4))
for row, (name, values, color) in enumerate([
    ('較穩定', steady, GREEN), ('較分散', variable, ORANGE)]):
    ax.scatter(values, np.full(len(values), row), s=70, color=color)
    print(f'{name}：平均 {values.mean():.1f}，標準差 {values.std(ddof=0):.2f}')
ax.axvline(20, color=PURPLE, linestyle='--', label='共同平均數：20')
ax.set(yticks=[0, 1], yticklabels=['較穩定', '較分散'],
       xlabel='配送時間（分鐘）', title='只看平均，會漏掉不穩定程度')
ax.legend()
plt.show()

## 03｜已被標記的產品，有多少真的有瑕疵？

**為什麼做**：先用人數表理解條件機率。假設 10,000 件產品中 1% 有瑕疵；模型抓到 90% 瑕疵，也誤標 5% 良品。

**怎麼看**：90 件瑕疵被標記，495 件良品也被標記；分母要改成所有被標記的產品，不能沿用全體或全部瑕疵。

In [ ]:
counts = pd.DataFrame([[9405, 495], [10, 90]],
    index=['實際良品', '實際瑕疵'], columns=['未標記', '被標記'])
display(counts)
posterior = counts.loc['實際瑕疵', '被標記'] / counts['被標記'].sum()
print(f'被標記產品真正有瑕疵的比例：{posterior:.2%}')
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(counts, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax)
ax.set(title='漏報少，不代表每個警報都是真的', xlabel='模型結果', ylabel='真實狀態')
plt.show()
assert np.isclose(posterior, 90 / 585)

## 接回機器學習

貝氏更新是在改變「已知條件後的分母」。本例的基準比例與分類表都是假想設定，不是真實模型測試；模型的 recall 高，也可能有低 precision。

## 換你解讀

1. 為什麼加入一次延遲後，平均數比中位數變化大？
2. 平均時間相同，配送承諾的風險也相同嗎？
3. 「抓到九成瑕疵」為什麼不等於「九成警報是真的」？

### 請 AI 幫你做一個小實驗

請 AI 保持抓到瑕疵的比例與誤標比例不變，將瑕疵率改成 10%。先預測警報可信度如何變，再用計數表驗證。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)